In [1]:
"""
The purpose of this Jupyter notebook is to identify the most stable
of the four measurement types in the screen (nucleus, perinucleus, cell,
Voronoi tesselation).

To this end, a special metric called coefficient of variation (CV) is
used. It is defined as the ratio between the standard deviation and the
mean. By relating the standard deviation to the mean, the variability is
made scale-independent. This is important when comparing different
measurement types, which often have different scales.

For each control, the plate means are computed, following which the
cross-plate CV is computed. As a second step, the variability
information provided by the individual controls is aggregated via the
median. Lastly, the measurement type with the smallest median CV is
chosen.
"""

'\nThe purpose of this Jupyter notebook is to identify the most stable\nof the four measurement types in the screen (nucleus, perinucleus, cell,\nVoronoi tesselation).\n\nTo this end, a special metric called coefficient of variation (CV) is\nused. It is defined as the ratio between the standard deviation and the\nmean. By relating the standard deviation to the mean, the variability is\nmade scale-independent. This is important when comparing different\nmeasurement types, which often have different scales.\n\nFor each control, the plate means are computed, following which the\ncross-plate CV is computed. As a second step, the variability\ninformation provided by the individual controls is aggregated via the\nmedian. Lastly, the measurement type with the smallest median CV is\nchosen.\n'

In [7]:
import pandas as pd

### Data Loading and Preparation

In [ ]:
# Load the screen subset TSV file into a DataFrame
screen_subset_path = (
    "Dharmacon_pooled_G1_G2_screening_plates_subset_control-based_Z-"
    "scored.tsv"
)

screen_subset_df = pd.read_csv(
    screen_subset_path,
    sep="\t"
)

In [9]:
# Extract controls from the DataFrame
# Omit the control type "UNKNOWN" while doing so
controls_df = screen_subset_df[
    (screen_subset_df["WellType"] == "CONTROL")
    &
    (screen_subset_df["Name"] != "UNKNOWN")
]

In [10]:
# Determine the unique control names
unique_control_names = controls_df["Name"].drop_duplicates().to_list()

assert "UNKNOWN" not in unique_control_names, (
    "The 'UNKNOWN' control has not been successfully removed!"
)

### Computing Control Means per Plate

In [12]:
# The individual controls occur multiple times on each plate
# Thus, for each control, the mean intensity per plate is computed
# To this end, multi-level grouping has to be performed for the plate ID
# (`Barcode`) as well as the control name (`Name`)
grouped_df = controls_df.groupby(["Barcode", "Name"])

# Index the columns of interest (the different intensity measurements
# for eGFP and mCherry) and compute the mean for each intensity
# measurement separately
control_mean_int_per_plate_df = grouped_df[[
    "dIntensity_cPathogen_eMean_oCells_nZScore",
    "dIntensity_cPathogen_eMean_oNuclei_nZScore",
    "dIntensity_cPathogen_eMean_oPeriNuclei_nZScore",
    "dIntensity_cPathogen_eMean_oVoronoiCells_nZScore",
    "dIntensity_cLatePathogen_eMean_oCells_nZScore",
    "dIntensity_cLatePathogen_eMean_oNuclei_nZScore",
    "dIntensity_cLatePathogen_eMean_oPeriNuclei_nZScore",
    "dIntensity_cLatePathogen_eMean_oVoronoiCells_nZScore"
]].mean()

In [17]:
# Now, the computation of the cross-plate CV is turned to
# This involves two steps
# The first step consists of computing the cross-plate standard
# deviation as well as the mean for each control
control_stats_df = (
    control_mean_int_per_plate_df
    .groupby(level="Name")
    .agg(["mean", "std"])
)

In [21]:
# The second step involves computing the CV for each control and each
# measurement type
cv_df = (
    control_stats_df.xs("std", axis=1, level=1)
    /
    # Bear in mind that after Z-scoring, the mean can also be negative,
    # which is why the absolute mean is used
    control_stats_df.xs("mean", axis=1, level=1).abs()
)

In [23]:
median_cv = cv_df.median(axis=0)

In [24]:
print(median_cv)

dIntensity_cPathogen_eMean_oCells_nZScore               0.648701
dIntensity_cPathogen_eMean_oNuclei_nZScore              0.694687
dIntensity_cPathogen_eMean_oPeriNuclei_nZScore          0.602655
dIntensity_cPathogen_eMean_oVoronoiCells_nZScore        0.686205
dIntensity_cLatePathogen_eMean_oCells_nZScore           0.930166
dIntensity_cLatePathogen_eMean_oNuclei_nZScore          0.683303
dIntensity_cLatePathogen_eMean_oPeriNuclei_nZScore      0.841000
dIntensity_cLatePathogen_eMean_oVoronoiCells_nZScore    0.854315
dtype: float64
